# Study A granular invariance analysis

Use this notebook for paper-facing slices that should stay separate from the canonical `study_a_analysis.ipynb`.

Typical inputs:

- per-case delta export from `scripts/evaluation/export_invariance_case_deltas.py`
- optional controllability JSON from `scripts/evaluation/run_controllability_comparison.py`
- failure-card outputs for taxonomy labelling

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

RUNTIME_ROOT = Path.cwd().resolve().parents[0]
CASE_DELTAS_PATH = RUNTIME_ROOT / "metric-results" / "invariance_smoke" / "qwen3-lmstudio" / "study_a_case_deltas.json"
CONTROLLABILITY_PATH = RUNTIME_ROOT / "metric-results" / "controllability" / "qwen3-lmstudio" / "study_a_archived_smoke.json"

rows = json.loads(CASE_DELTAS_PATH.read_text(encoding="utf-8")) if CASE_DELTAS_PATH.exists() else []
df = pd.DataFrame(rows)
print(f"case rows: {len(df)}")
display(df.head() if not df.empty else pd.DataFrame())

In [ ]:
if not df.empty:
    summary = (
        df.groupby(["metric"], as_index=False)
        .agg(mean_delta=("delta", "mean"), median_delta=("delta", "median"), n=("id", "count"))
        .sort_values("metric")
    )
    display(summary)

    if "condition" not in df.columns and "strata" in df.columns:
        df["condition"] = df["strata"].apply(lambda value: value.get("condition") if isinstance(value, dict) else None)
        df["risk"] = df["strata"].apply(lambda value: value.get("risk") if isinstance(value, dict) else None)

    for metric_name in sorted(df["metric"].unique()):
        subset = df[df["metric"] == metric_name]
        plt.figure(figsize=(8, 4))
        subset["delta"].hist(bins=20)
        plt.title(f"Study A delta distribution: {metric_name}")
        plt.xlabel("variant - base")
        plt.ylabel("count")
        plt.tight_layout()
        plt.show()

    display(df.sort_values("delta", ascending=False).head(20))